In [ ]:
import torch
print("--- Step-by-Step Similarity Calculation Example ---")

# 1. Define Sample Tensors
# Let's imagine we have one image and three text captions.
# Each feature vector has a dimension of 2 for simplicity.

# Image features (1 image, 2 dimensions)
# Imagine this image is somewhat similar to text 1, and less to text 2 and 3.
image_features = torch.tensor([[0.8, 0.6]], dtype=torch.float32) # Already normalized
print(f"\nSample Image Features (normalized):\n{image_features}")

# Text features (3 captions, 2 dimensions)
# Each row is a feature vector for a text caption.
# These are also normalized for cosine similarity.
text_features = torch.tensor([
    [0.7, 0.7],  # Text 1: "a dog", somewhat similar to image
    [0.1, 0.9],  # Text 2: "a cat", not very similar
    [0.9, 0.1]   # Text 3: "a car", not very similar
], dtype=torch.float32)
print(f"Sample Text Features (normalized):\n{text_features}")

# 2. Transpose Text Features
# For matrix multiplication, we need text_features to be (dimensions, captions)
text_features_T = text_features.T
print(f"\nText Features Transposed (.T):\n{text_features_T}")

# 3. Perform Matrix Multiplication (Cosine Similarity)
# image_features (1, 2) @ text_features.T (2, 3) -> result (1, 3)
raw_similarity_scores = image_features @ text_features_T
print(f"\nRaw Similarity Scores (image_features @ text_features.T):\n{raw_similarity_scores}")

# 4. Scale the Scores (Multiply by 100.0)
# This is a learned temperature parameter in CLIP to sharpen predictions.
scaled_similarity_scores = 100.0 * raw_similarity_scores
print(f"\nScaled Similarity Scores (100.0 * raw_similarity_scores):\n{scaled_similarity_scores}")

# 5. Apply Softmax to get Probabilities
# Softmax converts scores into a probability distribution.
# dim=-1 means apply softmax across the last dimension (the text captions).
probabilities = scaled_similarity_scores.softmax(dim=-1)
print(f"\nFinal Probabilities (.softmax(dim=-1)):\n{probabilities}")

# 6. Find the top 1 prediction
values, indices = probabilities[0].topk(1)
print(f"\nTop 1 Prediction Value: {values.item():.4f}")
print(f"Top 1 Prediction Index: {indices.item()}")

# If you had actual text labels, you could map the index back:
text_labels = ["a dog", "a cat", "a car"]
print(f"Predicted Label: {text_labels[indices.item()]}")


### Explanation of the `similarity` Calculation with Example:

In the example code above, we simulated the process using simple 2-dimensional feature vectors for one image and three text captions.

1.  **`image_features` and `text_features`**: We start with normalized feature vectors. Normalization is crucial because the dot product of normalized vectors directly gives their cosine similarity.
    *   `image_features` represents our single input image.
    *   `text_features` represents the embeddings for our possible captions.

2.  **`text_features.T`**: We transpose the `text_features`. This reshapes it from `[number_of_captions, embedding_dimension]` to `[embedding_dimension, number_of_captions]`. This is necessary for the matrix multiplication to correctly compute the dot product of the image's embedding with each text embedding.

3.  **`image_features @ text_features.T`**: This is the core similarity calculation. It performs a matrix multiplication. The output is a tensor where each element is the cosine similarity between the `image_features` and one of the `text_features`.
    *   `[0.8, 0.6]` (image) multiplied by `[0.7, 0.7]` (text 1) gives `0.8*0.7 + 0.6*0.7 = 0.56 + 0.42 = 0.98` (high similarity)
    *   `[0.8, 0.6]` (image) multiplied by `[0.1, 0.9]` (text 2) gives `0.8*0.1 + 0.6*0.9 = 0.08 + 0.54 = 0.62` (moderate similarity)
    *   `[0.8, 0.6]` (image) multiplied by `[0.9, 0.1]` (text 3) gives `0.8*0.9 + 0.6*0.1 = 0.72 + 0.06 = 0.78` (moderate similarity)
    The raw similarity scores reflect these dot products.

4.  **`100.0 * ...`**: The raw similarity scores are then multiplied by `100.0`. In the actual CLIP model, this is typically a learned scalar value (often represented as `log_scale` which gets exponentiated to `tau`). Its purpose is to increase the magnitude of the differences between the similarity scores, making the model more confident and its predictions sharper after the softmax step.

5.  **`.softmax(dim=-1)`**: Finally, the `softmax` function is applied to these scaled scores. Softmax takes a vector of real numbers and transforms it into a probability distribution, where each value is between 0 and 1, and the sum of all values is 1. `dim=-1` specifies that this operation should be performed across the last dimension (the dimension corresponding to the different text captions). This gives us the final probabilities, indicating how likely the image matches each caption.

In the example, you can see how the highest raw similarity score (0.98 for 'a dog') becomes the highest probability after scaling and softmax, leading to the correct prediction.